# Stratification & Splitting

In [1]:
#pip install iterative-stratification

Note: you may need to restart the kernel to use updated packages.


In [23]:
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
import numpy as np
import pandas as pd

In [24]:
random_seed = 344

In [25]:
mis_df = pd.read_csv('mis_df.csv')

In [26]:
# code written with support from Claude AI in using iterstrat package

# labels for stratification
labels_for_strat = mis_df[['opinion_label', 'misinformation_label', 
                        'jennifer', 'nicole', 'rachelle', 'shiao-li']]

# first split in to train set - combined dev/test set: 60% - 40%
split_1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.4, random_state=random_seed)
train_idx, temp_idx = next(split_1.split(mis_df, labels_for_strat))

temp_df = mis_df.iloc[temp_idx]
temp_labels = labels_for_strat.iloc[temp_idx]

# second split dev set - test set: 50% - 50% 
split_2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=random_seed)
dev_idx, test_idx = next(split_2.split(temp_df, temp_labels))

# add column for split label
mis_df['split'] = 'train'
mis_df.iloc[temp_idx[dev_idx], mis_df.columns.get_loc('split')] = 'dev'
mis_df.iloc[temp_idx[test_idx], mis_df.columns.get_loc('split')] = 'test'

In [27]:
print(mis_df['split'].value_counts())

split
train    600
dev      200
test     200
Name: count, dtype: int64


### Check distributions in splits

In [28]:
for split in ['train', 'dev', 'test']:
    print(f"\n=== {split} ===")
    subset = mis_df[mis_df['split'] == split]
    print(subset['opinion_label'].value_counts(normalize=True))
    print(subset['misinformation_label'].value_counts(normalize=True))
    print(subset['annotator'].value_counts(normalize=True))


=== train ===
opinion_label
0    0.585
1    0.415
Name: proportion, dtype: float64
misinformation_label
0    0.741667
1    0.258333
Name: proportion, dtype: float64
annotator
rachelle         0.258333
shiao-li         0.231667
nicole           0.205000
jennifer         0.203333
jasmine          0.053333
Majority Vote    0.045000
ai               0.003333
Name: proportion, dtype: float64

=== dev ===
opinion_label
0    0.585
1    0.415
Name: proportion, dtype: float64
misinformation_label
0    0.74
1    0.26
Name: proportion, dtype: float64
annotator
rachelle         0.255
shiao-li         0.230
nicole           0.205
jennifer         0.200
jasmine          0.070
Majority Vote    0.040
Name: proportion, dtype: float64

=== test ===
opinion_label
0    0.585
1    0.415
Name: proportion, dtype: float64
misinformation_label
0    0.74
1    0.26
Name: proportion, dtype: float64
annotator
rachelle         0.260
shiao-li         0.235
jennifer         0.205
nicole           0.205
jasmine      

In [29]:
mis_df.to_csv('mis_df_with_splits.csv', index=False)

In [30]:
mis_df[mis_df['split'] == 'train'].to_csv('mis_df_train.csv', index=False)
mis_df[mis_df['split'] == 'dev'].to_csv('mis_df_dev.csv', index=False)
mis_df[mis_df['split'] == 'test'].to_csv('mis_df_test.csv', index=False)

In [ ]:
# check for any null values
train_df = mis_df[mis_df['split'] == 'train']
print(train_df.isnull().sum())

id                       0
text                     0
misinformation_label     0
opinion_label            0
all_caps                 0
exclamation_marks        0
hedging                  0
adjectives               0
unk                      0
annotator                0
text_length              0
exclamation_marks_bin    0
all_caps_bin             0
hedging_bin              0
adjectives_bin           0
unk_bin                  0
jennifer                 0
nicole                   0
rachelle                 0
shiao-li                 0
split                    0
dtype: int64


In [32]:
print(train_df[train_df['annotator'].isnull()])

Empty DataFrame
Columns: [id, text, misinformation_label, opinion_label, all_caps, exclamation_marks, hedging, adjectives, unk, annotator, text_length, exclamation_marks_bin, all_caps_bin, hedging_bin, adjectives_bin, unk_bin, jennifer, nicole, rachelle, shiao-li, split]
Index: []

[0 rows x 21 columns]
